# 深層学習DIA解析結果を前処理する

**対応記事**: [article-11-openms-preprocess.md](../blog/article-11-openms-preprocess.md) — OpenMS前処理  
**実行順序**: 11番目  
**所要時間**: 約5分

---

## このNotebookで行うこと

OpenMS + AlphaPeptDeepで深層学習を使って生成した**19,981タンパク質のマトリクス**を、統計解析に適した形に前処理します。Log2変換・欠損値フィルタリング・欠損値補完をPythonで実装し、Sageの結果（notebook_05c）と同じ形式の前処理済みデータを作成します。

- Log2変換で正規分布に近づける
- 有効値が少なすぎるタンパク質を除去（70%ルール）
- 検出限界以下の欠損値をPerseus互換のdownshift法で補完
- 統計解析用の`preprocessed_data_openms.csv`を出力

**⚠️ 注意**: このNotebookを実行する前に、notebook_10でOpenMSパイプラインを実行し、`protein_matrix_from_openms.csv`が生成されている必要があります。

## 前提条件

- [notebook_10_openms.ipynb](./notebook_10_openms.ipynb) が完了していること
- `results/protein_matrix_from_openms.csv` が存在すること
- Python環境が適切に設定されていること

## 1. 深層学習DIAの前処理意義

**【なぜ前処理が必要？】**

- **データ規模**: 19,981タンパク質（Sageの9.5倍）の高密度データ
- **生強度の問題**: 10⁶〜10⁹と桁数が大きく、分布が偏っている
- **欠損値の意味**: 検出されない＝存在しないではなく、検出限界以下の低発現
- **統計解析準備**: 正規分布に近い、欠損値のないマトリクスが必要

In [ ]:
import os       # ファイルパスの結合・操作に使う標準ライブラリ
import numpy as np   # 数値計算ライブラリ（log2変換・乱数生成などに使用）
import pandas as pd  # データフレーム操作ライブラリ（CSV読み書き・欠損値処理などに使用）
import matplotlib.pyplot as plt  # グラフ描画ライブラリ
import seaborn as sns  # 統計的可視化ライブラリ
from pathlib import Path  # パス操作ライブラリ

# Jupyter notebook での図のインライン表示設定
%matplotlib inline

# 警告表示を簡潔にする
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# --- パス ---
# 結果ファイルを格納するディレクトリへの相対パス（notebookから実行する場合の基準）
RESULTS_DIR = "../results"
FIG_DIR = f"{RESULTS_DIR}/figures"
# 前処理済みデータのCSVファイルパス（step_10のOpenMSで作成したタンパク質定量マトリクス）
INPUT_CSV = os.path.join(RESULTS_DIR, "protein_matrix_from_openms.csv")

# ディレクトリ作成
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f"結果ディレクトリ: {RESULTS_DIR}")
print(f"入力ファイル: {INPUT_CSV}")
print(f"図保存先: {FIG_DIR}")

In [ ]:
# --- Perseus互換パラメータ ---
# downshift: 有効値の平均から何SD下にシフトするか
#   → 2.4 = 「検出されなかったタンパク質は有効値の平均より2.4SD低い」と仮定
# width: 補完値の分布幅（SDの倍率）
#   → 0.3 = 元のSDの30%の幅で補完値をばらつかせる

# Perseus互換のdownshiftパラメータ: 有効値の平均から2.4SD下にシフトして補完値の中心を決める
DOWNSHIFT = 2.4
# Perseus互換のwidthパラメータ: 補完値のばらつきを元のSDの30%に設定する
WIDTH = 0.3

# 有効値の最低割合（論文の設定: 70%）
# この割合未満の有効値しかないタンパク質は信頼性が低いため除去する
VALID_RATIO = 0.70

print(f"【Perseus互換パラメータ】")
print(f"Downshift: {DOWNSHIFT} (検出限界以下の推定)")
print(f"Width: {WIDTH} (補完値のばらつき)")
print(f"有効値最低割合: {VALID_RATIO*100:.0f}% (品質フィルタ)")

# 本書で扱うデータ規模の説明
print(f"\n【深層学習DIAの前処理対象】")
print(f"対象タンパク質数: 約19,981個 (Sageの9.5倍)")
print(f"サンプル数: 32個 (Normal 16 + Tumor 16)")
print(f"データタイプ: 高密度プロテオミクスデータ")
print(f"前処理の意義: 統計解析・機械学習に適した形式に変換")

## 2. データ読み込みと概要確認

In [ ]:
# データファイルの存在確認
if os.path.exists(INPUT_CSV):
    print(f"✅ 入力ファイル確認: {INPUT_CSV}")
    file_size = os.path.getsize(INPUT_CSV) / (1024**2)  # MB
    print(f"   ファイルサイズ: {file_size:.1f} MB")
else:
    print(f"❌ 入力ファイルが見つかりません: {INPUT_CSV}")
    print(f"notebook_10_openms.ipynb を先に実行してください")
    
    # デモ用の仮想データ作成
    print(f"\n🔄 デモ用仮想データを作成します...")
    
    # 仮想サンプル名
    sample_names = []
    for i in range(1, 17):
        sample_names.extend([f"Patient_{i:02d}-N", f"Patient_{i:02d}-T"])
    
    # 仮想タンパク質ID (UniProt形式)
    n_proteins = 19981
    protein_ids = [f"P{10000 + i:05d}" for i in range(n_proteins)]
    
    # 仮想強度データ生成（対数正規分布）
    np.random.seed(42)
    data = np.random.lognormal(mean=15, sigma=2, size=(n_proteins, 32))
    
    # 一部を欠損値にする（約15%）
    missing_mask = np.random.random((n_proteins, 32)) < 0.15
    data[missing_mask] = np.nan
    
    # DataFrameとして作成
    virtual_df = pd.DataFrame(data, index=protein_ids, columns=sample_names)
    virtual_df.index.name = "Protein"
    
    # CSVとして保存
    virtual_df.to_csv(INPUT_CSV)
    print(f"✅ 仮想データ作成完了: {virtual_df.shape}")

In [ ]:
# CSVファイルを読み込み、1列目（タンパク質名）をインデックスに設定する
df = pd.read_csv(INPUT_CSV, index_col=0)
# インデックスの名前を"Protein"に設定（後の処理で参照しやすくするため）
df.index.name = "Protein"

print(f"【データ読み込み完了】")
print(f"データ形状: {df.shape[0]:,} タンパク質 × {df.shape[1]} サンプル")
print(f"メモリ使用量: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

# データの基本統計
print(f"\n【基本統計】")
print(f"最小値: {df.min().min():.2e}")
print(f"最大値: {df.max().max():.2e}")
print(f"中央値: {df.median().median():.2e}")
print(f"欠損値総数: {df.isnull().sum().sum():,}")
print(f"欠損率: {df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100:.1f}%")

In [ ]:
# サンプル名の末尾 -N / -T で Normal（正常組織） / Tumor（腫瘍組織） を判別
# リスト内包表記で、列名に"-N"を含むサンプル名だけを抽出する
normal_samples = [c for c in df.columns if "-N" in c]
# 列名に"-T"を含むサンプル名だけを抽出する
tumor_samples  = [c for c in df.columns if "-T" in c]
# Normal群とTumor群をまとめた辞書を作成（後のフィルタリングで群ごとに処理するため）
groups = {"Normal": normal_samples, "Tumor": tumor_samples}

# 読み込んだデータの概要を表示（行数=タンパク質数、列数=サンプル数）
print(f"【サンプル分類】")
print(f"Normal群: {len(normal_samples)}サンプル")
print(f"Tumor群: {len(tumor_samples)}サンプル")
print(f"合計: {len(normal_samples) + len(tumor_samples)}サンプル")

print(f"\nNormal samples: {normal_samples[:3]}...")
print(f"Tumor samples:  {tumor_samples[:3]}...")

# サンプル分類の妥当性確認
if len(normal_samples) + len(tumor_samples) != df.shape[1]:
    print(f"⚠️ サンプル分類に問題があります")
    unclassified = [c for c in df.columns if c not in normal_samples + tumor_samples]
    print(f"未分類サンプル: {unclassified}")
else:
    print(f"✅ サンプル分類正常")

## 3. Log2変換

**【Log2変換の目的】**

- **正規分布化**: 質量分析の強度値（10⁶〜10⁹）を正規分布に近づける
- **倍率表現**: 差1.0 = 2倍変化として直感的に理解しやすい
- **統計解析準備**: t検定などの統計手法は正規分布を仮定
- **可視化改善**: ヒートマップ等での色分布が自然になる

In [ ]:
# Log2変換前の分布可視化
print("【Log2変換前の分布確認】")

# 全サンプルの中央値の中央値を計算し、データが既にlog2スケールか判定する基準にする
# df.median()で各列（サンプル）の中央値を取得し、さらにその中央値を取る
median_val = df.median().median()
mean_val = df.mean().mean()

print(f"全体の中央値: {median_val:.2e}")
print(f"全体の平均値: {mean_val:.2e}")

# データ分布の可視化
plt.figure(figsize=(15, 5))

# サンプルデータ（計算効率のため最初の1000タンパク質）
sample_data = df.iloc[:1000].values.flatten()
sample_data = sample_data[~np.isnan(sample_data)]  # 欠損値除去

# 左: 生データのヒストグラム
plt.subplot(1, 3, 1)
plt.hist(sample_data, bins=50, alpha=0.7, color='blue')
plt.xlabel('Raw Intensity')
plt.ylabel('Frequency')
plt.title('変換前の強度分布')
plt.yscale('log')

# 中央: Log10でのヒストグラム
plt.subplot(1, 3, 2)
plt.hist(np.log10(sample_data), bins=50, alpha=0.7, color='green')
plt.xlabel('Log10(Intensity)')
plt.ylabel('Frequency')
plt.title('Log10変換後の分布')

# 右: Log2でのヒストグラム（予想）
plt.subplot(1, 3, 3)
plt.hist(np.log2(sample_data), bins=50, alpha=0.7, color='red')
plt.xlabel('Log2(Intensity)')
plt.ylabel('Frequency')
plt.title('Log2変換後の分布（予想）')

plt.tight_layout()
plt.show()

print(f"データ範囲: {sample_data.min():.2e} ~ {sample_data.max():.2e}")
print(f"桁数の幅: {np.log10(sample_data.max()) - np.log10(sample_data.min()):.1f} 桁")

In [ ]:
# Log2変換の実行
print("【Log2変換実行】")

# 中央値が100より大きい場合は生の強度値と判断してLog2変換を実行する
if median_val > 100:
    print(f"生強度データと判定（中央値: {median_val:.2e}）")
    
    # 変換前の統計を記録
    before_stats = {
        'min': df.min().min(),
        'max': df.max().max(),
        'median': df.median().median(),
        'mean': df.mean().mean()
    }
    
    # 値0をNaN（欠損値）に置換してからLog2変換する（log2(0)=-∞を避けるため）
    df_log2 = np.log2(df.replace(0, np.nan))
    
    # 変換後の統計
    after_stats = {
        'min': df_log2.min().min(),
        'max': df_log2.max().max(),
        'median': df_log2.median().median(),
        'mean': df_log2.mean().mean()
    }
    
    print(f"✅ Log2変換完了")
    print(f"変換前 - 最小値: {before_stats['min']:.2e}, 最大値: {before_stats['max']:.2e}")
    print(f"変換後 - 最小値: {after_stats['min']:.2f}, 最大値: {after_stats['max']:.2f}")
    print(f"中央値変化: {before_stats['median']:.2e} → {after_stats['median']:.2f}")
    
    # dfを更新
    df = df_log2
    
else:
    # 中央値が100以下の場合、既にlog2スケールに変換済みと判断してスキップする
    print(f"既にlog2スケール（中央値: {median_val:.1f}）→ スキップ")

# 変換後のデータ確認
print(f"\n変換後データ:")
print(f"形状: {df.shape}")
print(f"欠損値: {df.isnull().sum().sum():,}個")
print(f"値の範囲: {df.min().min():.2f} ~ {df.max().max():.2f}")

## 4. 有効値フィルタリング（70%ルール）

**【70%ルールとは？】**

- **定義**: 各群（Normal/Tumor）で少なくとも70%のサンプルで検出されたタンパク質のみ残す
- **目的**: 統計検定に十分なデータ数を確保、信頼性の低いタンパク質を除去
- **判定**: いずれかの群で基準を満たせばOK（AND条件ではなくOR条件）
- **根拠**: 論文の標準的設定、統計的パワーの確保

In [ ]:
def filter_by_valid_ratio(df, groups, ratio=VALID_RATIO):
    """いずれかの群で有効値割合 >= ratio を満たすタンパク質を残す。

    【なぜフィルタリングが必要か？】
      - ほとんどのサンプルで検出されないタンパク質は統計的に信頼できない
      - 欠損値が多すぎると補完の精度も低下する
      - 統計検定には十分なサンプル数が必要
      
    Parameters
    ----------
    df : pd.DataFrame
        タンパク質定量データ（行=タンパク質、列=サンプル）
    groups : dict
        群別サンプル名辞書 {'Normal': [...], 'Tumor': [...]}
    ratio : float
        有効値最低割合（0.7 = 70%）
        
    Returns
    -------
    pd.DataFrame
        フィルタリング後のデータ
    """
    # 全タンパク質をFalse（除去対象）で初期化する。条件を満たしたものだけTrueにする
    keep = pd.Series(False, index=df.index)
    
    print(f"【各群の有効値割合確認】")
    
    # 各群（Normal, Tumor）について有効値割合を確認するループ
    for group_name, samples in groups.items():
        # notna()で欠損でないセルをTrue/Falseに変換し、行方向に合計して有効値数を求める
        # それをサンプル数で割って有効値の「割合」を計算する（0.0〜1.0の範囲）
        valid_counts = df[samples].notna().sum(axis=1)
        valid_ratios = valid_counts / len(samples)
        
        # 基準を満たすタンパク質数
        passed = (valid_ratios >= ratio).sum()
        
        print(f"{group_name}群 ({len(samples)}サンプル):")
        print(f"  基準値以上: {passed:,}タンパク質 ({passed/len(df)*100:.1f}%)")
        print(f"  平均有効率: {valid_ratios.mean()*100:.1f}%")
        print(f"  有効率範囲: {valid_ratios.min()*100:.1f}% ~ {valid_ratios.max()*100:.1f}%")
        
        # OR演算（|=）で「いずれかの群で基準を満たせばTrue」にする
        # これにより、Normal群またはTumor群のどちらかで70%以上有効なら残す
        keep |= (valid_ratios >= ratio)
    
    # keepがTrueのタンパク質だけを残したDataFrameを返す
    return df[keep], keep.sum()

# フィルタリング前のタンパク質数を記録しておく（除去数の表示に使う）
n_before = len(df)
print(f"フィルタリング前: {n_before:,}タンパク質")

# 70%ルールでフィルタリングを実行し、結果をdfに上書きする
df_filtered, n_kept = filter_by_valid_ratio(df, groups)

print(f"\n【フィルタリング結果】")
print(f"残存: {n_kept:,}タンパク質")
print(f"除去: {n_before - n_kept:,}タンパク質")
print(f"残存率: {n_kept/n_before*100:.1f}%")

# データを更新
df = df_filtered

## 5. 欠損値補完（Perseus互換 downshift法）

**【Perseus downshift法とは？】**

- **基本思想**: 検出されない＝存在しないではなく、検出限界以下の低発現
- **downshift**: 検出された値の平均よりdownshift×SD分だけ低い値で補完
- **width**: 補完値にも自然なばらつきを与える（元SDのwidth倍）
- **Perseus互換**: プロテオミクス解析の標準ツールと同じアルゴリズム

In [ ]:
def impute_downshift(df, downshift=DOWNSHIFT, width=WIDTH):
    """Perseus互換: 各サンプルごとに低値側から欠損を補完する。

    【補完の考え方】
      「検出されなかった」= 存在しないのではなく「低すぎて検出限界以下」
      → 検出された値の分布から推測して低い値を割り当てる

    【手順（各サンプルごと）】
      1. 有効値の平均(μ)と標準偏差(σ)を計算
      2. 補完値の中心 = μ - downshift × σ
      3. 補完値の幅 = width × σ
      4. この分布からランダムにサンプリング
      
    Parameters
    ----------
    df : pd.DataFrame
        フィルタリング済みデータ
    downshift : float
        補完値を有効値平均から何SD下にシフトするか
    width : float
        補完値のばらつき（元SDに対する倍率）
        
    Returns
    -------
    tuple
        (補完済みDataFrame, 補完値総数)
    """
    # 元のDataFrameを変更しないようにコピーを作成する
    df = df.copy()
    # 補完した値の総数をカウントする変数（最後に報告用に使う）
    total_imputed = 0
    imputation_stats = []  # 各サンプルの補完統計
    
    print(f"【各サンプルの欠損値補完】")
    
    # 各サンプル（列）を順番に処理するループ
    for i, col in enumerate(df.columns):
        # そのサンプルの有効値（NaNでない値）だけを取り出す
        valid = df[col].dropna()
        # そのサンプルでNaN（欠損値）の位置をTrue/Falseのマスクとして取得する
        mask = df[col].isna()
        # 欠損値の個数を数える
        n_missing = mask.sum()
        
        # 有効値が0個の場合は補完できないのでスキップする
        if len(valid) == 0 or n_missing == 0:
            continue
            
        # 補完値の中心: 有効値の平均からdownshift×SD分だけ低い値にする
        # 例: 平均20, SD=2, downshift=2.4 → 中心は 20 - 2.4×2 = 15.2
        imp_mean = valid.mean() - downshift * valid.std()
        # 補完値のばらつき: 元のSDのwidth倍（0.3なら30%）に設定する
        imp_std  = width * valid.std()
        
        # 欠損値が1個以上ある場合のみ補完を実行する
        if n_missing > 0:
            # 正規分布N(imp_mean, imp_std)からn個の乱数を生成し、欠損位置に代入する
            imputed_values = np.random.normal(imp_mean, imp_std, n_missing)
            df.loc[mask, col] = imputed_values
            # 補完した個数を累計に加算する
            total_imputed += n_missing
            
            # 統計記録
            stats = {
                'sample': col,
                'valid_count': len(valid),
                'missing_count': n_missing,
                'valid_mean': valid.mean(),
                'valid_std': valid.std(),
                'imp_mean': imp_mean,
                'imp_std': imp_std
            }
            imputation_stats.append(stats)
        
        # 進捗表示（5サンプルごと）
        if (i + 1) % 8 == 0 or i == len(df.columns) - 1:
            print(f"  {i+1:2d}/{len(df.columns)} 完了 - {col}: {len(valid)}有効, {n_missing}補完")
    
    # 補完統計のサマリー
    if imputation_stats:
        stats_df = pd.DataFrame(imputation_stats)
        print(f"\n【補完統計サマリー】")
        print(f"総補完値数: {total_imputed:,}")
        print(f"平均補完数/サンプル: {stats_df['missing_count'].mean():.1f}")
        print(f"補完率範囲: {stats_df['missing_count'].min()}-{stats_df['missing_count'].max()}個/サンプル")
    
    # 補完済みDataFrameと補完値の総数を返す
    return df, total_imputed

# 再現性のため乱数シードを固定する（42は慣例的によく使われる値）
np.random.seed(42)
print(f"乱数シード固定: 42（再現性確保）")

# 補完前の欠損値統計
missing_before = df.isnull().sum().sum()
print(f"\n補完前欠損値: {missing_before:,}個")

# downshift法で欠損値を補完し、補完後のデータと補完数を受け取る
df_imputed, n_imputed = impute_downshift(df)

# 結果表示
missing_after = df_imputed.isnull().sum().sum()
print(f"\n【補完結果】")
print(f"補完値数: {n_imputed:,}")
print(f"残り欠損: {missing_after}個")
print(f"補完パラメータ: downshift={DOWNSHIFT}, width={WIDTH}")

if missing_after == 0:
    print(f"✅ 全ての欠損値が補完されました")
else:
    print(f"⚠️ {missing_after}個の欠損値が残っています")

# データを更新
df = df_imputed

## 6. 前処理結果の可視化と検証

In [ ]:
# 前処理結果の可視化
plt.figure(figsize=(15, 10))

# 上段: データ分布の変化
plt.subplot(2, 3, 1)
sample_data = df.iloc[:1000].values.flatten()
sample_data = sample_data[~np.isnan(sample_data)]
plt.hist(sample_data, bins=50, alpha=0.7, color='green')
plt.xlabel('Log2(Intensity)')
plt.ylabel('Frequency')
plt.title('前処理後の強度分布')

# 上段中: サンプル間相関
plt.subplot(2, 3, 2)
# 計算効率のため最初の5サンプルの相関
corr_matrix = df.iloc[:500, :5].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0.8)
plt.title('サンプル間相関（最初5サンプル）')
plt.tight_layout()

# 上段右: 群別分布比較
plt.subplot(2, 3, 3)
normal_data = df[normal_samples].iloc[:500].values.flatten()
tumor_data = df[tumor_samples].iloc[:500].values.flatten()
normal_data = normal_data[~np.isnan(normal_data)]
tumor_data = tumor_data[~np.isnan(tumor_data)]

plt.hist(normal_data, bins=30, alpha=0.5, label='Normal', color='blue')
plt.hist(tumor_data, bins=30, alpha=0.5, label='Tumor', color='red')
plt.xlabel('Log2(Intensity)')
plt.ylabel('Frequency')
plt.title('群別強度分布比較')
plt.legend()

# 下段: 品質指標
plt.subplot(2, 3, 4)
# サンプルごとの統計
sample_stats = df.describe().T
plt.scatter(sample_stats['mean'], sample_stats['std'], alpha=0.7)
plt.xlabel('Mean Log2 Intensity')
plt.ylabel('Standard Deviation')
plt.title('サンプル別統計分布')

# 下段中: タンパク質ごとのCV分布
plt.subplot(2, 3, 5)
protein_cv = (df.std(axis=1) / df.mean(axis=1)) * 100
protein_cv = protein_cv[~np.isnan(protein_cv)]
plt.hist(protein_cv, bins=50, alpha=0.7, color='orange')
plt.xlabel('CV (%)')
plt.ylabel('Frequency')
plt.title('タンパク質別CV分布')
plt.axvline(protein_cv.median(), color='red', linestyle='--', 
           label=f'Median: {protein_cv.median():.1f}%')
plt.legend()

# 下段右: 前処理サマリー
plt.subplot(2, 3, 6)
# 前処理ステップの情報をテキストで表示
summary_text = f"""前処理完了サマリー

最終データ形状:
{df.shape[0]:,} タンパク質 × {df.shape[1]} サンプル

処理ステップ:
✓ Log2変換
✓ 70%フィルタリング
✓ Perseus互換補完

品質指標:
・欠損値: {df.isnull().sum().sum()}個
・平均CV: {protein_cv.mean():.1f}%
・値範囲: {df.min().min():.1f}~{df.max().max():.1f}

出力ファイル:
preprocessed_data_openms.csv
sample_info.csv"""

plt.text(0.05, 0.95, summary_text, transform=plt.gca().transAxes, 
         fontsize=9, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
plt.axis('off')
plt.title('前処理サマリー')

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/openms_preprocessing_summary.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"前処理可視化を保存しました: {FIG_DIR}/openms_preprocessing_summary.png")

## 7. データ保存

In [ ]:
# 前処理済みのタンパク質定量マトリクスをCSVファイルとして保存する
output_file = os.path.join(RESULTS_DIR, "preprocessed_data_openms.csv")
df.to_csv(output_file)

print(f"【前処理済みデータ保存】")
print(f"ファイル: {output_file}")
print(f"サイズ: {os.path.getsize(output_file) / (1024**2):.1f} MB")
print(f"形状: {df.shape[0]:,} タンパク質 × {df.shape[1]} サンプル")

# 最終データの品質確認
print(f"\n【最終品質確認】")
print(f"欠損値: {df.isnull().sum().sum()}個")
print(f"無限値: {np.isinf(df.values).sum()}個")
print(f"値の範囲: {df.min().min():.2f} ~ {df.max().max():.2f}")
print(f"平均値: {df.mean().mean():.2f}")
print(f"標準偏差: {df.std().mean():.2f}")

In [ ]:
# サンプル情報（各サンプルがNormalかTumorか）をDataFrameにまとめる
# 後続のステップ（差分解析・可視化）で群の情報が必要になるため、別ファイルとして保存する
sample_info = pd.DataFrame({
    # サンプル名のリスト: Normal群とTumor群を結合する
    "Sample": normal_samples + tumor_samples,
    # 条件ラベル: Normal群のサンプル数分"Normal"、Tumor群のサンプル数分"Tumor"を生成する
    "Condition": ["Normal"] * len(normal_samples) + ["Tumor"] * len(tumor_samples)
})

# 患者IDの抽出（Patient_XX部分）
sample_info["Patient_ID"] = sample_info["Sample"].str.extract(r'(Patient_\d+)')

# サンプル情報をCSVファイルとして保存する（index=Falseで行番号を出力しない）
sample_info_file = os.path.join(RESULTS_DIR, "sample_info_openms.csv")
sample_info.to_csv(sample_info_file, index=False)

print(f"\n【サンプル情報保存】")
print(f"ファイル: {sample_info_file}")
print(f"サンプル情報:")
print(sample_info.head(10))

print(f"\n【条件別集計】")
condition_counts = sample_info["Condition"].value_counts()
print(condition_counts)

# 保存完了のメッセージを表示して、最終的なデータサイズを確認する
print(f"\n✅ 前処理完了: {df.shape[0]:,} タンパク質 × {df.shape[1]} サンプル")
print(f"📁 出力ファイル:")
print(f"  → preprocessed_data_openms.csv ({os.path.getsize(output_file) / (1024**2):.1f} MB)")
print(f"  → sample_info_openms.csv ({os.path.getsize(sample_info_file) / 1024:.1f} KB)")

## 8. Sageとの比較

In [ ]:
# Sage前処理結果との比較（存在する場合）
sage_file = os.path.join(RESULTS_DIR, "preprocessed_data.csv")

if os.path.exists(sage_file):
    print("【Sage結果との比較】")
    
    try:
        sage_df = pd.read_csv(sage_file, index_col=0)
        
        comparison_data = {
            "指標": [
                "タンパク質数",
                "サンプル数", 
                "総データ点数",
                "欠損値",
                "値の範囲（Log2）",
                "平均値",
                "標準偏差",
                "ファイルサイズ"
            ],
            "Sage": [
                f"{sage_df.shape[0]:,}",
                f"{sage_df.shape[1]}",
                f"{sage_df.shape[0] * sage_df.shape[1]:,}",
                f"{sage_df.isnull().sum().sum()}",
                f"{sage_df.min().min():.1f} ~ {sage_df.max().max():.1f}",
                f"{sage_df.mean().mean():.2f}",
                f"{sage_df.std().mean():.2f}",
                f"{os.path.getsize(sage_file) / (1024**2):.1f} MB"
            ],
            "OpenMS深層学習": [
                f"{df.shape[0]:,}",
                f"{df.shape[1]}",
                f"{df.shape[0] * df.shape[1]:,}",
                f"{df.isnull().sum().sum()}",
                f"{df.min().min():.1f} ~ {df.max().max():.1f}",
                f"{df.mean().mean():.2f}",
                f"{df.std().mean():.2f}",
                f"{os.path.getsize(output_file) / (1024**2):.1f} MB"
            ]
        }
        
        comparison_df = pd.DataFrame(comparison_data)
        
        print(comparison_df.to_string(index=False))
        
        # 改善率計算
        protein_improvement = df.shape[0] / sage_df.shape[0]
        data_improvement = (df.shape[0] * df.shape[1]) / (sage_df.shape[0] * sage_df.shape[1])
        
        print(f"\n【性能向上率】")
        print(f"タンパク質数: {protein_improvement:.1f}倍向上")
        print(f"データ密度: {data_improvement:.1f}倍向上")
        print(f"深層学習の効果: 約{protein_improvement:.1f}倍の高感度検出を実現")
        
    except Exception as e:
        print(f"Sageファイル読み込みエラー: {e}")
else:
    print("Sage前処理結果が見つかりません（notebook_05cを先に実行）")
    print("\n【OpenMS深層学習の特徴】")
    print(f"・約19,981タンパク質の高密度検出")
    print(f"・従来手法（Sage）の約9.5倍の感度")
    print(f"・低発現タンパク質の包括的同定")
    print(f"・バイオマーカー発見の新可能性")

## まとめ

このNotebookでは以下のOpenMS前処理を実行しました：

1. **Log2変換**: 生強度値（10⁶〜10⁹）を正規分布に近い形に変換
2. **70%フィルタリング**: いずれかの群で70%以上のサンプルで検出されたタンパク質のみ残存
3. **Perseus互換補完**: downshift法で検出限界以下の欠損値を統計的に補完
4. **品質検証**: 統計解析に適したクリーンなデータの確認

**深層学習DIAの優位性:**
- **高密度データ**: 約19,981タンパク質（従来の9.5倍）
- **高品質**: 欠損値0、正規分布に近い分布
- **統計解析準備**: t検定、PCA、機械学習に最適
- **生物学的意義**: 低発現バイオマーカーの包括的検出

**出力ファイル:**
- `preprocessed_data_openms.csv`: 前処理済み高密度プロテオミクスデータ
- `sample_info_openms.csv`: サンプル条件情報

**次のステップ:**
この高品質な前処理済みデータを用いて、従来手法では不可能だった詳細な統計解析・可視化・バイオマーカー探索を実行します。

---

## Navigation

⬅️ **前回**: [notebook_10_openms.ipynb](./notebook_10_openms.ipynb) — OpenMS深層学習パイプライン  
➡️ **次回**: [notebook_12a_openms_visualization.ipynb](./notebook_12a_openms_visualization.ipynb) — OpenMS可視化

---

*このNotebookは [article-11-openms-preprocess.md](../blog/article-11-openms-preprocess.md) に対応しています。*